# De Bruijn graphs

In [ ]:
"""
project04.ipynb

Implements a De Bruijn graph genome assembler to reconstruct a genome from
short sequencing reads. read_fastq reads the sequences from a FASTQ file.
The DeBruijnGraph class stores the graph as an adjacency list (self.graph),
where each node is a (k-1)-mer and each k-mer is a directed edge from its
left (k-1)-mer to its right (k-1)-mer. build_graph_from_reads adds every
read and its reverse complement to the graph, storing each distinct edge
once, using add_edge. eulerian_walk recursively walks a copy of the graph,
removing each edge it uses along with its twin edge on the opposite strand,
and returns the tour in reverse order. assemble_contigs starts walks from
nodes with more outgoing than incoming edges, then from any node with edges
remaining, splits each tour wherever neighboring nodes are not connected in
the original graph, and converts each piece to a DNA sequence using
tour_to_sequence. get_assembly_stats calculates the number of contigs,
total, longest, shortest, and mean contig length, and N50. write_fasta
writes the contigs to a FASTA file. The recursion limit is raised to
50,000,000 to allow the deep recursive walks required by the yeast genome.

For the Toy Example section, the class is tested on 9 overlapping 10 bp
reads with k=6, which assemble into a single 26 bp contig.

For the Real Data section, the driver program assembles the Saccharomyces
cerevisiae R64 genome from 2,414,266 simulated, error-free 150 bp
single-end reads at 30x coverage (data/sacCer_R64_150SE_30x_wgsim.fq.gz)
with k=31, and writes the contigs to genome_assembly.fasta and the
statistics to genome_assembly_stats.txt.

Authors are Matthew Francoeur and Ildiko Polyak.
"""

In [1]:
import gzip


def read_fastq(filename):
    """Read sequences from a FASTQ file.

    Args:
        filename (str): Path to FASTQ file.

    Returns:
        list: List of DNA sequence strings (quality scores ignored).

    Example:
        >>> reads = read_fastq("test_reads.fastq")
        >>> len(reads)
        10
    """

    opener = gzip.open if filename.endswith(".gz") else open
    sequences = []
    with opener(filename, "rt") as fasta:
        for line_count, line in enumerate(fasta, 1):
            if line_count % 4 == 2:
                sequences.append(line.strip().upper())
    return sequences

In [1]:
"""De Bruijn Graph Genome Assembly Module.

This module provides classes and functions for constructing De Bruijn graphs
from sequencing reads and performing genome assembly using Eulerian path
traversal.
"""
from collections import defaultdict
import random
import sys
sys.setrecursionlimit(50000000)   # raised above the ~23 million edges, so no walk can exceed it


class DeBruijnGraph:
    """Main class for De Bruijn graphs and genome assembly.

    This class builds De Bruijn graphs from sequencing reads and performs
    genome assembly by finding Eulerian paths through connected components
    of the graph.

    Attributes:
        graph (defaultdict): Adjacency list representation of graph edges.
            Keys are (k-1)-mers, values are lists of adjacent (k-1)-mers.
        k (int): The k-mer size used for graph construction.

    Example:
        >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
        >>> dbg = DeBruijnGraph(reads, k=6)
        >>> contigs = dbg.assemble_contigs(seed=42)
        >>> len(contigs) > 0
        True
    """

    def __init__(self, reads, k):
        """Initialize De Bruijn graph from sequencing reads.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer size for graph construction.

        Example:
            >>> reads = ["ATGGCG", "GCGTGC", "TGCAAC"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> len(dbg.graph) > 0
            True
        """
        self.graph = defaultdict(list)
        self.k = k
        self.build_graph_from_reads(reads, k)


    def add_edge(self, left, right):
        """Add a directed edge to the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> "TGG" in dbg.graph["ATG"]
            True
        """
        #add right to the end of left's list in self.graph
        self.graph[left].append(right)
        
    def remove_edge(self, left, right):
        """Remove a directed edge from the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> dbg.remove_edge("ATG", "TGG")
            >>> len(dbg.graph["ATG"])
            0
        """
        #remove right from left's list in self.graph
        self.graph[left].remove(right)

    def build_graph_from_reads(self, reads, k):
        """Build De Bruijn graph from multiple sequencing reads.

        Extracts all k-mers from all reads and adds edges between
        consecutive (k-1)-mers within each k-mer.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer length for graph construction.

        Example:
            >>> reads = ["ATGGC", "TGGCA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.build_graph_from_reads(reads, 4)
            >>> "ATG" in dbg.graph
            True
        """
        #Sanity-check print statement:  print how many reads there are and the value of k
        #print(f"Building graph from {len(reads)} reads with k = {k}")

        complement = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}
        both_strands = []
        for read in reads:
            rc_read = ""
            for base in reversed(read):
                rc_read = rc_read + complement[base]
            both_strands.append(read)
            both_strands.append(rc_read)

        for read in both_strands:   #FOR each read in on both strands:
            for i in range(len(read) - k + 1):  #FOR each position, from 0 up to (length of read - k):
                kmer = read[i:i + k]  #kmer = the k letters starting at this position
                left =  kmer[:-1]  #left = kmer without its last letter
                right = kmer[1:]  #right = kmer without its first letter
                #if read == reads[0]:  #Sanity-check print statement: print each kmer with its left and right node, first read only
                    #print(f"kmer: {kmer}   left: {left}   right: {right}")
                if right not in self.graph[left]:  #IF right is not already in left's list in self.graph:
                    self.add_edge(left, right)  #self.add_edge(left, right)
        #num_edges = 0  #Sanity-check print statement: pring number of nodes, number of edges
        #for node in self.graph:
            #num_edges = num_edges + len(self.graph[node])
        #print(f"Graph built: {len(self.graph)} nodes, {num_edges} edges")


    def eulerian_walk(self, node, graph, seed=None):
        """Perform recursive Eulerian walk on a graph component.

        This is a recursive function that follows all edges from a node
        to traverse the graph, building a path in reverse order.

        Args:
            node (str): Current node to traverse from.
            graph (defaultdict): Graph or subgraph to traverse.
            seed (int, optional): Seed for random edge selection.

        Returns:
            list: List of (k-1)-mers traversed (in reverse order).

        Example:
            >>> reads = ["ATGGCG"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> graph_copy = defaultdict(list, dbg.graph)
            >>> tour = dbg.eulerian_walk("ATG", graph_copy, seed=42)
            >>> len(tour) > 0
            True
        """
        if seed is not None:  #IF a seed was given:
            random.seed(seed)  #set the random seed (only in 1st call)
        tour = []   #tour = an empty list
        complement = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}
        while len(graph[node]) > 0:  #WHILE node's list in graph is not empty:
            next_node = random.choice(graph[node])  #next node = pick one of node's list in graph at random
            graph[node].remove(next_node)  #remove next node from node's list in graph (the copy)
            twin_left = ""
            for base in reversed(next_node):
                twin_left = twin_left + complement[base]
            twin_right = ""
            for base in reversed(node):
                twin_right = twin_right + complement[base]
            if twin_right in graph.get(twin_left, []):
                graph[twin_left].remove(twin_right)
            sub_tour = self.eulerian_walk(next_node, graph) 
            if len(tour) == 0:  
                tour = sub_tour  #nothing to join yet, so reuse it (no copying)
            else:  #only happens at branch points
                tour.extend(sub_tour)  #add sub_tour onto the end of tour
        tour.append(node)  #add node to end of tour
        return tour  #Return tour
    

    def assemble_contigs(self, seed=None):
        """Assemble all contigs from the De Bruijn graph.

        Finds all connected components and generates an Eulerian path
        for each component, producing multiple assembled contigs.

        Args:
            seed (int, optional): Random seed for reproducible assembly.

        Returns:
            list: List of assembled contig sequences (DNA strings).

        Example:
            >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
            >>> dbg = DeBruijnGraph(reads, k=6)
            >>> contigs = dbg.assemble_contigs(seed=42)
            >>> all(isinstance(c, str) for c in contigs)
            True
        """
        if seed is not None:   #IF a seed was given:
            random.seed(seed)  #set the random seed (once for all walks)
        copy = defaultdict(list)  #copy = a new empty graph
        for node in self.graph:  #FOR each node in self.graph:
            copy[node] = list(self.graph[node])  #copy's list for node = a copy of node's list
        #print(f"Nodes in self.graph:  {len(self.graph)}  Nodes in copy:  {len(copy)}")  #Sanity-check print statement: print node count in self.graph and in copy (should match)

        arriving = defaultdict(int)   #arriving = a count for each node, starting at 0
        for node in self.graph:  #FOR each node in self.graph:
            for neighbor in self.graph[node]:  #FOR each neighbor in node's list:
                arriving[neighbor] = arriving[neighbor] + 1  #add 1 to arriving for neighbor
        
        start_nodes = []  #an empty list to collect them in
        for node in self.graph:
            if len(self.graph[node]) > arriving[node]:  #leaving > arriving
                start_nodes.append(node)  #add it to the list
        #print(f"Start nodes found:  {len(start_nodes)}")  #Sanity-check print statement: print how many start nodes were found

        all_nodes = list(self.graph)  #all nodes = a list of every node in self.graph, made now, before any walking 
        contigs = []   #contigs = an empty list
        kept = set()   #kept = an empty set
        #walk_count = 0

        for node in start_nodes + all_nodes:  #FOR each node in (start nodes, followed by all nodes):
            if len(copy[node]) > 0:  #IF node's list in copy is not empty:
                tour = self.eulerian_walk(node, copy)  #tour = self.eulerian_walk (node, copy)
                #walk_count = walk_count + 1  #count this walk
                #if walk_count <= 3:  #show the tour length
                    #print (f"Walk {walk_count}:  tour length {len(tour)}")  #Sanity-check print statement: print the length of the tour (first 3 walks only)

                tour.reverse() #now it reads start to end
                pieces = [[tour[0]]]  #pieces = a list holding one piece: [the first node]
                for step in tour [1:]:  #go through every node after the first
                    previous = pieces[-1][-1]  #previous = the last node of the current piece
                    if step in self.graph.get(previous, []):  #is the previous step a real edge in the original?
                        pieces[-1].append(step)  #yes: add it to current piece
                    else:
                        pieces.append([step])  #no, a false join: start a new piece holding just this node
                                
                complement = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"} #dictionary of base pairs
                for piece in pieces:  #go through each piece
                    if len(piece) >= 2:  #IF the piece has at least 2 nodes (skip one node pieces, which have no edges)
                        sequence = self.tour_to_sequence(piece) #uses tour_to_sequence() function
                        rc = ""  #builds the reverse complement
                        for base in reversed(sequence):
                            rc = rc + complement[base]
                        if sequence not in kept and rc not in kept:  #neither one has been kept yet
                            contigs.append(sequence)  #add it to results
                            kept.add(sequence)  #remember it (sets use .add, lists use .append)
                        
        #print(f"Contigs assembled: {len(contigs)}")  #Sanity check print statement: print number of contigs
        #edges_after = 0
        #for n in self.graph:
            #edges_after = edges_after + len(self.graph[n])
        #print(f"Edges in self.graph after assembly:  {edges_after} (should match 'Graph built')")  #Sanity check print statement: print edge count in self.graph (should be unchanged, proving original is intact)

        return contigs

    def tour_to_sequence(self, tour):
        """Convert a tour of (k-1)-mers into a DNA sequence.

        Args:
            tour (list): List of (k-1)-mer strings in order.

        Returns:
            str: Assembled DNA sequence.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> tour = ['ATG', 'TGG', 'GGC', 'GCG']
            >>> dbg.tour_to_sequence(tour)
            'ATGGCG'
        """
        sequence = tour[0]  #sequence = the whole first node in tour
        for node in tour[1:]:   #FOR each node after the first one:
            sequence = sequence + node[-1]  #add that node's last letter to the end of sequence
        return sequence   #Return sequence

    def get_assembly_stats(self, contigs):
        """Calculate assembly statistics for assembled contigs.

        Args:
            contigs (list): List of contig sequences.

        Returns:
            dict: Dictionary containing assembly statistics:
                - num_contigs: Total number of contigs
                - total_length: Total assembled sequence length
                - longest_contig: Length of longest contig
                - shortest_contig: Length of shortest contig
                - mean_length: Mean contig length
                - n50: N50 statistic

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA", "CCCCCCCCCC"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> stats = dbg.get_assembly_stats(contigs)
            >>> stats['num_contigs']
            3
        """
        lengths = []  #lengths = a list of the length of each contig
        for contig in contigs:
            lengths.append(len(contig))

        if len(contigs) == 0:  #IF there are no contigs:
            return {"num_contigs": 0, "total_length": 0, "longest_contig": 0, "shortest_contig": 0, "mean_length": 0, "n50": 0}   #Return all six stats set to 0

        num_contigs = len(contigs)  #num_contigs = how many contigs are there
        total_length = sum(lengths)  #total_length = all lengths added together
        longest_contig = max(lengths)  #longest_contig = the biggest length
        shortest_contig = min(lengths)  #shortest_contig = the smallest length
        mean_length = total_length / num_contigs #mean_length = total_length / num_contigs

        sorted_lengths = sorted(lengths, reverse=True)  #sort the lengths, longest first
        running_total = 0  #start running total at 0
        n50 = 0  #starting value so n50 always exists
        for length in sorted_lengths:
            running_total = running_total + length  #add it to the running total
            if running_total >= total_length / 2:  #reached halfway yet?
                n50 = length  #record it, then stop
                break
        #Return a dictionary holding the six stats
        return {"num_contigs": num_contigs, "total_length": total_length, "longest_contig": longest_contig, "shortest_contig": shortest_contig, "mean_length": mean_length, "n50": n50}  

        
    def write_fasta(self, contigs, filename):
        """Write assembled contigs to a FASTA file.

        Args:
            contigs (list): List of contig sequences.
            filename (str): Output FASTA filename.

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.write_fasta(contigs, "output.fasta")
        """

        with open(filename, "w") as f:  #open the file called filename, for writing ("with" will close file)
            number = 0
            for contig in contigs:  #number the contigs
                number = number + 1
                f.write(f">contig_{number} length={len(contig)}\n")  #write ">contig_" + number + " length=" + length of contig, then a line break
                f.write(contig + "\n")  #write the contig, then a line break
        #print(f"Wrote {len(contigs)} contigs to {filename}")  #Sanity-print check statement:  print the filename and how many contigs were written

        
      
                    
                    

                


---
# Toy Example

In [4]:
print("="*60)
print("EXAMPLE 1: Assembling 9 overlapping reads into one contig")
print("="*60)

toy_reads_1 = [
    "ATGGCGTACG",  # Read 1
    "GGCGTACGTT",  # Read 2: overlaps with Read 1
    "CGTACGTTAC",  # Read 3: overlaps with Read 2
    "TACGTTACCA",  # Read 4: overlaps with Read 3
    "CGTTACCATG",  # Read 5: overlaps with Read 4
    "TTACCATGGG",  # Read 6: overlaps with Read 5
    "ACCATGGGCC",  # Read 7: overlaps with Read 6
    "CATGGGCCTA",  # Read 8: overlaps with Read 7
    "TGGGCCTAAA"   # Read 9: overlaps with Read 8
]

print(f"\nInput: {len(toy_reads_1)} reads")
print("First read:  ", toy_reads_1[0])
print("Last read:   ", toy_reads_1[-1])
print(f"\nBuilding De Bruijn graph with k=6...")
dbg = DeBruijnGraph(toy_reads_1, k=6)
contigs = dbg.assemble_contigs(seed=42)
print(contigs)
print(dbg.get_assembly_stats(contigs))

EXAMPLE 1: Assembling 9 overlapping reads into one contig

Input: 9 reads
First read:   ATGGCGTACG
Last read:    TGGGCCTAAA

Building De Bruijn graph with k=6...
['ATGGCGTACGTTACCATGGGCCTAAA']
{'num_contigs': 1, 'total_length': 26, 'longest_contig': 26, 'shortest_contig': 26, 'mean_length': 26.0, 'n50': 26}


---
# Real data
This section utilizes a FASTQ file with 2.4 million "perfect" 150 bp reads simulated
from the _S. cerevisiae_ (yeast) genome (`R64`) and tasks your program to ingest, assemble, and traverse
the genome with statistical output report.

In [4]:
"""Driver program for genome assembly using De Bruijn graphs.

This cell runs the completed DeBruijnGraph class on a FASTQ file of perfect
(error-free, variant-free) single-end reads simulated from a reference genome,
then reports assembly statistics.

Usage:
    Run all cells above this one first, then set the configuration values
    below and run this cell.

Expected input:
    - FASTQ or gzipped FASTQ (.fq, .fastq, .fq.gz, .fastq.gz)
    - Perfect single-end reads simulated with wgsim

Output:
    - <output_fasta>: Assembled contigs
    - <stats_file>: Detailed statistics report
"""

import time
from datetime import datetime

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
INPUT_FILE = "data/sacCer_R64_150SE_30x_wgsim.fq.gz"
OUTPUT_FASTA = "genome_assembly.fasta"
STATS_FILE = "genome_assembly_stats.txt"
K_MER_SIZE = 31
# Size (bp) of the reference the reads were simulated from; used for the
# coverage and assembly-fraction estimates. Set to None if unknown.
GENOME_SIZE = 12_071_326


def _pct(part, whole):
    """Return part/whole as a percentage, or 0.0 when whole is zero."""
    return (part / whole * 100) if whole else 0.0


def write_statistics_file(
    stats_file,
    input_file,
    num_reads,
    avg_read_length,
    k_mer_size,
    num_nodes,
    num_edges,
    stats,
    contig_lengths,
    timing,
    genome_size,
    coverage_estimate,
    assembly_fraction,
):
    """Write comprehensive assembly statistics to a text file.

    Args:
        stats_file (str): Output filename.
        input_file (str): Input FASTQ filename.
        num_reads (int): Number of reads processed.
        avg_read_length (float): Average read length.
        k_mer_size (int): K-mer size used.
        num_nodes (int): Number of graph nodes.
        num_edges (int): Number of distinct graph edges.
        stats (dict): Assembly statistics.
        contig_lengths (list): Contig lengths sorted longest first.
        timing (dict): Timing information, including 'total_time'.
        genome_size (int or None): Reference genome size in bp.
        coverage_estimate (float or None): Estimated sequencing coverage.
        assembly_fraction (float or None): Assembly size as % of genome.
    """
    rule = "-" * 80 + "\n"
    banner = "=" * 80 + "\n"

    with open(stats_file, "w") as f:
        f.write(banner)
        f.write("GENOME ASSEMBLY STATISTICS\n")
        f.write(banner + "\n")

        f.write(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input file:    {input_file}\n")
        f.write(f"K-mer size:    {k_mer_size}\n\n")

        f.write(rule + "INPUT DATA\n" + rule)
        f.write(f"Number of reads:         {num_reads:,}\n")
        f.write(f"Average read length:     {avg_read_length:.1f} bp\n")
        f.write(f"Total sequencing data:   {num_reads * avg_read_length:,.0f} bp\n")
        if genome_size:
            f.write(f"Reference genome size:   {genome_size:,} bp\n")
            f.write(f"Estimated coverage:      {coverage_estimate:.1f}x\n")
        f.write(f"Read time:               {timing['read_time']:.2f} seconds\n\n")

        f.write(rule + "DE BRUIJN GRAPH CONSTRUCTION\n" + rule)
        f.write(f"Graph nodes:             {num_nodes:,}\n")
        f.write(f"Graph edges:             {num_edges:,}\n")
        avg_degree = num_edges / num_nodes if num_nodes else 0.0
        f.write(f"Average out-degree:      {avg_degree:.2f}\n")
        f.write(f"Construction time:       {timing['graph_time']:.2f} seconds\n\n")

        f.write(rule + "ASSEMBLY RESULTS\n" + rule)
        f.write(f"Number of contigs:       {stats['num_contigs']:,}\n")
        f.write(f"Total assembly length:   {stats['total_length']:,} bp\n")
        if genome_size:
            f.write(f"Assembly vs. genome:     {assembly_fraction:.2f}%\n")
        f.write(f"Longest contig:          {stats['longest_contig']:,} bp\n")
        f.write(f"Shortest contig:         {stats['shortest_contig']:,} bp\n")
        f.write(f"Mean contig length:      {stats['mean_length']:,.1f} bp\n")
        f.write(f"N50:                     {stats['n50']:,} bp\n")
        f.write(f"Assembly time:           {timing['assembly_time']:.2f} seconds\n\n")

        f.write(rule + "TOP 20 LONGEST CONTIGS\n" + rule)
        for i, length in enumerate(contig_lengths[:20], 1):
            f.write(f"{i:3d}. {length:10,} bp\n")
        f.write("\n")

        f.write(rule + "CONTIG LENGTH DISTRIBUTION\n" + rule)
        bins = [
            ("all", len(contig_lengths)),
            (">100kb", sum(1 for x in contig_lengths if x > 100_000)),
            (">50kb", sum(1 for x in contig_lengths if x > 50_000)),
            (">10kb", sum(1 for x in contig_lengths if x > 10_000)),
            (">5kb", sum(1 for x in contig_lengths if x > 5_000)),
            (">1kb", sum(1 for x in contig_lengths if x > 1_000)),
            (">500bp", sum(1 for x in contig_lengths if x > 500)),
        ]
        for bin_name, count in bins:
            f.write(f"Contigs {bin_name:8s}:     {count:,}\n")
        f.write("\n")

        total_time = timing["total_time"]
        f.write(rule + "TIMING SUMMARY (excluding file output)\n" + rule)
        for label, key in (
            ("Read time:", "read_time"),
            ("Graph construction:", "graph_time"),
            ("Assembly:", "assembly_time"),
        ):
            f.write(f"{label:25s}{timing[key]:8.2f} seconds "
                    f"({_pct(timing[key], total_time):5.1f}%)\n")
        f.write(f"{'Total time:':25s}{total_time:8.2f} seconds "
                f"({total_time / 60:.2f} minutes)\n\n")

        f.write(banner + "END OF REPORT\n" + banner)


def count_graph(graph):
    """Count nodes and distinct edges in an adjacency-list graph.

    Nodes that only receive edges may not be keys of the adjacency dict, so
    they are counted separately. Duplicate neighbor entries (the same k-mer
    seen in several reads) are counted once.

    Args:
        graph (dict): Mapping of (k-1)-mer -> iterable of neighbor (k-1)-mers.

    Returns:
        tuple: (num_nodes, num_edges)
    """
    num_edges = 0
    sink_only = set()
    for neighbors in graph.values():
        distinct = set(neighbors)
        num_edges += len(distinct)
        for node in distinct:
            if node not in graph:  # membership test does not insert keys
                sink_only.add(node)
    return len(graph) + len(sink_only), num_edges


def assemble_genome(
    input_file=INPUT_FILE,
    output_fasta=OUTPUT_FASTA,
    stats_file=STATS_FILE,
    k_mer_size=K_MER_SIZE,
    genome_size=GENOME_SIZE,
):
    """Run the complete assembly pipeline.

    1. Load FASTQ reads
    2. Build the De Bruijn graph
    3. Assemble contigs
    4. Calculate statistics
    5. Write output files

    Returns:
        dict: Assembly results with keys 'dbg', 'contigs', 'stats', 'timing',
            'graph_stats', 'coverage', 'output_fasta', and 'stats_file'.
    """
    print("=" * 80)
    print("GENOME ASSEMBLY PIPELINE")
    print("=" * 80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    timing = {}

    # ------------------------------------------------------------------ 1
    print("STEP 1: Loading sequencing reads")
    print("-" * 80)
    print(f"Input file: {input_file}\n")

    start_time = time.time()
    reads = read_fastq(input_file)
    timing["read_time"] = time.time() - start_time

    num_reads = len(reads)
    if num_reads == 0:
        raise ValueError(f"No reads found in {input_file}")
    total_bases = sum(len(read) for read in reads)
    avg_read_length = total_bases / num_reads

    print(f"Reads loaded: {num_reads:,}")
    print(f"Total bases: {total_bases:,} bp")
    print(f"Average read length: {avg_read_length:.1f} bp")
    print(f"Time elapsed: {timing['read_time']:.2f} seconds")
    print("\nSample reads:")
    print(f"  First: {reads[0][:80]}...")
    print(f"  Last:  {reads[-1][:80]}...\n")

    # ------------------------------------------------------------------ 2
    print("STEP 2: Building De Bruijn graph")
    print("-" * 80)
    print(f"K-mer size: {k_mer_size}")
    print(f"Building graph from {num_reads:,} reads...\n")

    start_time = time.time()
    dbg = DeBruijnGraph(reads, k=k_mer_size)
    timing["graph_time"] = time.time() - start_time

    del reads  # the graph holds everything needed; free the read list

    num_nodes, num_edges = count_graph(dbg.graph)
    avg_degree = num_edges / num_nodes if num_nodes else 0.0

    print("Graph construction complete!")
    print(f"  Nodes (distinct {k_mer_size - 1}-mers): {num_nodes:,}")
    print(f"  Edges (distinct {k_mer_size}-mers):   {num_edges:,}")
    print(f"  Average out-degree: {avg_degree:.2f}")
    print(f"Time elapsed: {timing['graph_time']:.2f} seconds\n")

    # ------------------------------------------------------------------ 3
    print("STEP 3: Assembling contigs")
    print("-" * 80)
    print("Traversing the graph to build contigs...\n")

    start_time = time.time()
    contigs = dbg.assemble_contigs()
    timing["assembly_time"] = time.time() - start_time
    timing["total_time"] = (
        timing["read_time"] + timing["graph_time"] + timing["assembly_time"]
    )

    print("Assembly complete!")
    print(f"  Contigs generated: {len(contigs):,}")
    print(f"Time elapsed: {timing['assembly_time']:.2f} seconds\n")

    # ------------------------------------------------------------------ 4
    print("STEP 4: Calculating assembly statistics")
    print("-" * 80)

    stats = dbg.get_assembly_stats(contigs)
    contig_lengths = sorted((len(c) for c in contigs), reverse=True)

    print("Assembly statistics:")
    print(f"  Number of contigs:     {stats['num_contigs']:,}")
    print(f"  Total assembly length: {stats['total_length']:,} bp")
    print(f"  Longest contig:        {stats['longest_contig']:,} bp")
    print(f"  Shortest contig:       {stats['shortest_contig']:,} bp")
    print(f"  Mean contig length:    {stats['mean_length']:,.1f} bp")
    print(f"  N50:                   {stats['n50']:,} bp\n")

    print("Top 10 longest contigs:")
    for i, length in enumerate(contig_lengths[:10], 1):
        print(f"  {i:2d}. {length:,} bp")
    print()

    if genome_size:
        coverage_estimate = total_bases / genome_size
        assembly_fraction = _pct(stats["total_length"], genome_size)
        print("Genome coverage analysis:")
        print(f"  Reference genome size:         {genome_size:,} bp")
        print(f"  Estimated sequencing coverage: {coverage_estimate:.1f}x")
        print(f"  Assembly size vs. genome:      {assembly_fraction:.1f}%")
        print("  (Near 200% usually means reverse-complement duplicate "
              "contigs were not removed.)\n")
    else:
        coverage_estimate = assembly_fraction = None

    # ------------------------------------------------------------------ 5
    print("STEP 5: Writing output files")
    print("-" * 80)

    dbg.write_fasta(contigs, output_fasta)
    print(f"✓ Contigs written to: {output_fasta}")

    write_statistics_file(
        stats_file,
        input_file,
        num_reads,
        avg_read_length,
        k_mer_size,
        num_nodes,
        num_edges,
        stats,
        contig_lengths,
        timing,
        genome_size,
        coverage_estimate,
        assembly_fraction,
    )
    print(f"✓ Statistics written to: {stats_file}\n")

    print("=" * 80)
    print("ASSEMBLY COMPLETE")
    print("=" * 80)
    print(f"Total time (excluding file output): {timing['total_time']:.2f} seconds "
          f"({timing['total_time'] / 60:.2f} minutes)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    return {
        "dbg": dbg,
        "contigs": contigs,
        "stats": stats,
        "timing": timing,
        "graph_stats": {
            "num_nodes": num_nodes,
            "num_edges": num_edges,
            "avg_degree": avg_degree,
        },
        "coverage": coverage_estimate,
        "output_fasta": output_fasta,
        "stats_file": stats_file,
    }


print("\n" + "=" * 80)
print("GENOME DE BRUIJN GRAPH ASSEMBLY")
print("Student Assignment Driver Program")
print("=" * 80 + "\n")

result = assemble_genome()

print("=" * 80)
print("FIVE LONGEST ASSEMBLED CONTIGS")
print("=" * 80 + "\n")

for i, contig in enumerate(sorted(result["contigs"], key=len, reverse=True)[:5], 1):
    preview = contig[:100] + "..." if len(contig) > 100 else contig
    print(f">contig_{i} length={len(contig)}")
    print(preview + "\n")

print("=" * 80)
print("✓ ASSEMBLY COMPLETE")
print("=" * 80)
print("\nResults saved to:")
print(f"  • {result['output_fasta']} - {result['stats']['num_contigs']:,} assembled contigs")
print(f"  • {result['stats_file']} - detailed assembly statistics")
print("\nKey metrics:")
print(f"  • Total assembly: {result['stats']['total_length']:,} bp")
print(f"  • N50: {result['stats']['n50']:,} bp")
print(f"  • Longest contig: {result['stats']['longest_contig']:,} bp")
if result["coverage"] is not None:
    print(f"  • Coverage: {result['coverage']:.1f}x")
print()


GENOME DE BRUIJN GRAPH ASSEMBLY
Student Assignment Driver Program

GENOME ASSEMBLY PIPELINE
Start time: 2026-10-13 05:43:35

STEP 1: Loading sequencing reads
--------------------------------------------------------------------------------
Input file: data/sacCer_R64_150SE_30x_wgsim.fq.gz

Reads loaded: 2,414,266
Total bases: 362,139,900 bp
Average read length: 150.0 bp
Time elapsed: 3.91 seconds

Sample reads:
  First: ATCACAAAGGAGACAAAATCGTTGATAAAAAGTGCATCAACATTCCCAGCCAGAAAATGCACATCATAAAGACATGTTAT...
  Last:  ATTTAGCTCGCTTGTTTCTTGATTTTTTAGTCTCAATTTCATATCGTTAATAAAATCCTTGTGAGCTTCTGAGTTTAAAG...

STEP 2: Building De Bruijn graph
--------------------------------------------------------------------------------
K-mer size: 31
Building graph from 2,414,266 reads...

Graph construction complete!
  Nodes (distinct 30-mers): 22,953,466
  Edges (distinct 31-mers):   22,961,448
  Average out-degree: 1.00
Time elapsed: 366.01 seconds

STEP 3: Assembling contigs
------------------------------------